# **Day 69: Testing with pytest and doctest**

**pytest** is the most popular testing tool in Python. It does the same job as `unittest`, but it is simpler:
- No class is needed.
- We just use the normal `assert` keyword.

Install it with `pip install pytest`.

## **The Code to Test**

In [1]:
%%writefile landcover.py
def classify_ndvi(value):
    """Classify an NDVI value.

    >>> classify_ndvi(-0.3)
    'Water'
    >>> classify_ndvi(0.1)
    'Bare'
    >>> classify_ndvi(0.35)
    'Sparse vegetation'
    >>> classify_ndvi(0.8)
    'Dense vegetation'
    >>> classify_ndvi(1.5)
    Traceback (most recent call last):
    ...
    ValueError: NDVI must be between -1 and 1
    """
    if not -1 <= value <= 1:
        raise ValueError("NDVI must be between -1 and 1")
    if value < 0:
        return "Water"
    if value < 0.2:
        return "Bare"
    if value < 0.5:
        return "Sparse vegetation"
    return "Dense vegetation"


def split_bill(total, people, tip_percent=0):
    if people < 1:
        raise ValueError("People must be at least 1")
    return round(total * (1 + tip_percent / 100) / people, 2)

Writing landcover.py


## **Writing pytest Tests**

In [2]:
%%writefile test_landcover.py
import pytest
from landcover import classify_ndvi, split_bill

def test_water():
    assert classify_ndvi(-0.5) == "Water"

@pytest.mark.parametrize("value, expected", [
    (-1.0, "Water"),
    (0.0, "Bare"),
    (0.19, "Bare"),
    (0.2, "Sparse vegetation"),
    (0.49, "Sparse vegetation"),
    (0.5, "Dense vegetation"),
    (1.0, "Dense vegetation"),
])
def test_all_classes(value, expected):
    assert classify_ndvi(value) == expected

def test_out_of_range():
    with pytest.raises(ValueError, match="between -1 and 1"):
        classify_ndvi(2)

def test_split_bill():
    assert split_bill(900, 3) == 300

def test_split_with_tip():
    assert split_bill(1000, 4, tip_percent=10) == 275

@pytest.fixture
def phuchka_order():
    return {"plates": 4, "price": 30}

def test_with_fixture(phuchka_order):
    total = phuchka_order["plates"] * phuchka_order["price"]
    assert split_bill(total, 2) == 60

Writing test_landcover.py


In [3]:
import pytest
pytest.main(["-v", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings", "test_landcover.py"])

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Manas\AppData\Local\Python\pythoncore-3.14-64\Scripts\python.exe
rootdir: G:\My Drive\Python_programming\100-days-of-python\04_Advanced_Python
plugins: platformdirs-4.12.0
collecting ... 

collected 12 items

test_landcover.py::test_water 

PASSED                                     [  8%]


test_landcover.py::test_all_classes[-1.0-Water] 

PASSED                   [ 16%]


test_landcover.py::test_all_classes[0.0-Bare] 

... (17 outputs left out)


test_landcover.py::test_with_fixture 

PASSED                              [100%]



============================= 12 passed in 0.29s ==============================


<ExitCode.OK: 0>

Let us understand the code above:
- `@pytest.mark.parametrize` runs the **same test with many inputs**. Here one test function became 7 tests. This is great for checking the **boundary values** (like exactly 0.2 and 0.5), where bugs often hide.
- `pytest.raises` checks that an error is raised.
- A **fixture** (`@pytest.fixture`) prepares some data and gives it to any test that asks for it by name.

In the terminal, we simply run:
```
pytest -v
```

## **What a Failure Looks Like in pytest**

In [4]:
%%writefile test_fail_demo.py
def average(nums):
    return sum(nums) / len(nums) + 1       # Bug

def test_average():
    assert average([30, 32, 34]) == 32

Writing test_fail_demo.py


In [5]:
pytest.main(["-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings", "test_fail_demo.py"])

F

                                                                        [100%]


================================== FAILURES ===================================
________________________________ test_average _________________________________

    def test_average():
>       assert average([30, 32, 34]) == 32
E       assert 33.0 == 32
E        +  where 33.0 = average([30, 32, 34])

test_fail_demo.py:5: AssertionError
=========================== short test summary info ===========================
FAILED test_fail_demo.py::test_average - assert 33.0 == 32
1 failed in 0.25s


<ExitCode.TESTS_FAILED: 1>

pytest shows the actual values (`33.0 == 32`), which makes the bug easy to find.

## **doctest: Tests Inside the Docstring**
Look at the docstring of `classify_ndvi` above. It contains examples written like a Python session (`>>>`). The `doctest` module **runs these examples** and checks that the output matches.

So the documentation is **also** a test, and it can never become out of date.

In [6]:
import doctest, landcover
doctest.testmod(landcover, verbose=True)

Trying:
    classify_ndvi(-0.3)
Expecting:
    'Water'
ok
Trying:
    classify_ndvi(0.1)
Expecting:
    'Bare'
ok
Trying:
    classify_ndvi(0.35)
Expecting:
    'Sparse vegetation'
ok
Trying:
    classify_ndvi(0.8)
Expecting:
    'Dense vegetation'
ok
Trying:
    classify_ndvi(1.5)
Expecting:
    Traceback (most recent call last):
    ...
    ValueError: NDVI must be between -1 and 1
ok
2 items had no tests:
    landcover
    landcover.split_bill
1 item passed all tests:
   5 tests in landcover.classify_ndvi
5 tests in 3 items.
5 passed.
Test passed.


TestResults(failed=0, attempted=5)

## **Test Driven Development (TDD) in 3 Steps**
1. **Write a test first** for the new feature. It fails, because the feature does not exist yet.
2. **Write the simplest code** that makes the test pass.
3. **Improve** the code, while the tests keep it safe.

In [7]:
# Step 1: The test
def test_is_palindrome():
    assert is_palindrome("Malayalam")
    assert is_palindrome("A man a plan a canal Panama")
    assert not is_palindrome("Kolkata")

# Step 2: The code
def is_palindrome(s):
    s = "".join(ch.lower() for ch in s if ch.isalnum())
    return s == s[::-1]

# Step 3: Run the test
test_is_palindrome()
print("Test passed")

Test passed


In [8]:
# Quick question
# How many tests will pytest run for a parametrize list with 7 cases?